# FLOPs-budgeted HPO

ASHA searches learning rate and weight decay for the selected NATS-Bench architectures. Reusable experiment code lives in `kd_vs_hpo.hpo`.

In [ ]:
from pathlib import Path

import plotly.express as px
import torch

from kd_vs_hpo.common import TrainConfig
from kd_vs_hpo.hpo import ASHAConfig, HPOExperimentConfig, SearchSpace, run_hpo_experiment

%load_ext autoreload
%autoreload 2

In [ ]:
experiment = HPOExperimentConfig(
    train=TrainConfig(
        batch_size=256,
        num_workers=2,
        validation_fraction=0.1,
        momentum=0.9,
        grad_clip_norm=5.0,
        seed=42,
        deterministic=False,
        amp=True,
        train_step_multiplier=3.0,
        data_root=Path("data"),
    ),
    search_space=SearchSpace(
        lr=(1e-3, 3e-1),
        weight_decay=(1e-6, 1e-3),
    ),
    asha=ASHAConfig(
        budget_flops_per_arch=10**15,
        target_min_epochs=3,
        reduction_factor=3,
        max_initial_configs=12,
        max_epochs=81,
    ),
    architectures_path=Path("experiments/nats_architectures_10.json"),
    costs_path=Path("experiments/sampled_architecture_costs.csv"),
    output_dir=Path("hpo_output"),
    arch_rows=(0,),  # None runs all architectures.
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
result = run_hpo_experiment(experiment, device)
result.summary

In [ ]:
completed = result.stages[result.stages["status"] == "completed"].copy()
completed["budget_used_pct"] = 100 * completed["cumulative_flops"] / completed["budget_flops"]

px.line(
    completed,
    x="budget_used_pct",
    y="best_val_acc1_so_far",
    color="arch_index",
    markers=True,
    hover_data=["trial_id", "rung", "target_epochs", "lr", "weight_decay", "val_acc1"],
    title="Best validation accuracy under the FLOPs budget",
).show()